# Clase 157 — Variational Autoencoders (VAE)

El **VAE** (Kingma & Welling, 2014) es la versión probabilística del AE: el
encoder devuelve `(μ, log σ²)` de una gaussiana, se muestrea `z` con el
**reparametrization trick** `z = μ + σ·ε`, y la loss es el **ELBO**
= `reconstruction + β·KL(q(z|x) ‖ N(0,I))`. Resultado: latent continuo → permite
generar e interpolar.

**Requiere:** `tensorflow` / `keras`. El código es la **API real de Keras**
(capa custom + `train_step`); no se ejecuta sin TF. La fórmula de KL se ilustra
además en numpy.

## 1. Entorno

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
    tf.random.set_seed(42)
    print('tensorflow:', tf.__version__)
except Exception as e:
    HAS_TF = False
    print('tensorflow no instalado. Motivo:', type(e).__name__)

import numpy as np
np.random.seed(42)
latent_dim = 2

## 2. Capa de muestreo (reparametrization trick)

`z = μ + exp(0.5·log σ²)·ε`, con `ε ~ N(0, I)`. El ruido `ε` es externo, así los
gradientes fluyen por `μ` y `σ`.

In [ ]:
if HAS_TF:
    class Sampling(layers.Layer):
        def call(self, inputs):
            z_mean, z_log_var = inputs
            eps = tf.random.normal(shape=tf.shape(z_mean))
            return z_mean + tf.exp(0.5 * z_log_var) * eps
    print('Capa Sampling definida (reparametrization trick).')
else:
    # ilustracion numpy del truco
    z_mean, z_log_var = np.zeros(4), np.zeros(4)
    eps = np.random.normal(size=4)
    z = z_mean + np.exp(0.5 * z_log_var) * eps
    print('z (numpy) =', np.round(z, 3))

## 3. Encoder: `x → (z_mean, z_log_var) → z`

Con la Functional API se devuelven las tres salidas.

In [ ]:
if HAS_TF:
    enc_in = keras.Input(shape=(784,))
    h = layers.Dense(256, activation='relu')(enc_in)
    z_mean = layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(h)
    z = Sampling()([z_mean, z_log_var])
    encoder = keras.Model(enc_in, [z_mean, z_log_var, z], name='encoder')
    encoder.summary()
else:
    print('Encoder: 784->256->(z_mean, z_log_var)->Sampling->z')

## 4. Decoder: `z → x_reconstruido`

In [ ]:
if HAS_TF:
    dec_in = keras.Input(shape=(latent_dim,))
    d = layers.Dense(256, activation='relu')(dec_in)
    dec_out = layers.Dense(784, activation='sigmoid')(d)
    decoder = keras.Model(dec_in, dec_out, name='decoder')
    decoder.summary()
else:
    print('Decoder: z(latent_dim)->256->784(sigmoid)')

## 5. Modelo VAE con `train_step` custom (ELBO)

`KL(N(μ,σ²) ‖ N(0,I)) = -0.5·Σ(1 + log σ² - μ² - σ²)`. La loss total es
`reconstruction + KL` y se optimiza con un `GradientTape`.

In [ ]:
if HAS_TF:
    class VAE(keras.Model):
        def __init__(self, encoder, decoder, **kw):
            super().__init__(**kw)
            self.encoder, self.decoder = encoder, decoder

        def train_step(self, data):
            with tf.GradientTape() as tape:
                z_mean, z_log_var, z = self.encoder(data)
                recon = self.decoder(z)
                rec_loss = tf.reduce_sum(
                    keras.losses.binary_crossentropy(
                        data[..., None], recon[..., None]), axis=-1)
                kl = -0.5 * tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
                loss = tf.reduce_mean(rec_loss + kl)
            grads = tape.gradient(loss, self.trainable_weights)
            self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
            return {'loss': loss, 'kl': tf.reduce_mean(kl)}

    vae = VAE(encoder, decoder)
    vae.compile(optimizer='adam')
    print('VAE compilado. Loss = BCE_reconstruccion + KL (ELBO).')
else:
    mu, logv = np.array([0.5, -0.3]), np.array([0.1, -0.2])
    kl = -0.5 * np.sum(1 + logv - mu**2 - np.exp(logv))
    print('KL(numpy) =', round(float(kl), 4))

## 6. Generación: muestrear `z ~ N(0,I)` → decoder

Para generar imágenes nuevas se muestrea del **prior** (no del posterior).

In [ ]:
z_samples = np.random.normal(size=(64, latent_dim)).astype('float32')
if HAS_TF:
    generated = decoder.predict(z_samples, verbose=0)
    print('generadas:', generated.shape)   # (64, 784)
else:
    print('decoder.predict(z ~ N(0,I))  -> 64 imagenes nuevas de 784 pixeles')

# Interpolacion lineal en el latente entre dos puntos z_A, z_B
z_A, z_B = z_samples[0], z_samples[1]
alphas = np.linspace(0, 1, 10)[:, None]
z_interp = (1 - alphas) * z_A + alphas * z_B
print('interpolacion:', z_interp.shape)   # (10, latent_dim) -> transiciones suaves

## Ejercicios

1. **VAE básico**: entrená el VAE en MNIST y verificá que la `loss` baja.
2. **Sampling**: muestreá `z ~ N(0,I)` de tamaño `(100, latent_dim)` y visualizá el grid.
3. **Interpolación**: interpolá entre `z_A` y `z_B` (10 pasos) y mostrá transiciones suaves.
4. **β-VAE**: multiplicá el término KL por `β ∈ {1, 5, 10}` y compará disentanglement vs blur.
5. **Posterior collapse**: con LR alto, verificá `z_mean.std() → 0` (el encoder colapsa).

## Conclusiones

- El VAE aprende una **distribución** `(μ, σ)` sobre el latente, no un punto.
- El reparametrization trick `z = μ + σ·ε` hace diferenciable el muestreo.
- La loss ELBO = `reconstruction + β·KL` equilibra fidelidad y estructura latente.
- Un latente continuo permite **generar** (muestrear del prior) e **interpolar**.
- Los outputs salen borrosos (MSE/BCE) → GANs y difusión mejoran la nitidez.